In [14]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv('./Churn_Modelling.csv')

In [3]:
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

le = LabelEncoder()
df['Geography'] = le.fit_transform(df['Geography'])
df['Gender'] = le.fit_transform(df['Gender'])

X = df.drop('Exited', axis=1)
y = df['Exited']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB()
}

results = []

# Train, evaluate and save models
for name, model in models.items():

    # Train model
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Store results
    results.append([name, accuracy, precision, recall, f1])

    # Save model
    if name == "Logistic Regression":
        joblib.dump(model, "logistic_model.pkl")
    elif name == "KNN":
        joblib.dump(model, "knn_model.pkl")
    elif name == "Naive Bayes":
        joblib.dump(model, "naive_bayes_model.pkl")

# Save scaler
joblib.dump(scaler, "scaler.pkl")

# Create comparison table
comparison_df = pd.DataFrame(
    results,
    columns=["Algorithm", "Accuracy", "Precision", "Recall", "F1 Score"]
)

comparison_df = comparison_df.round(4)

print("\nComparison Table")
print(comparison_df)

# Best algorithm
best = comparison_df.loc[comparison_df["Accuracy"].idxmax()]

print("\nBest Algorithm:", best["Algorithm"])
print("Best Accuracy:", best["Accuracy"])


Comparison Table
             Algorithm  Accuracy  Precision  Recall  F1 Score
0  Logistic Regression    0.8155     0.6000  0.1832    0.2807
1                  KNN    0.8355     0.6333  0.3868    0.4803
2          Naive Bayes    0.8285     0.6812  0.2392    0.3540

Best Algorithm: KNN
Best Accuracy: 0.8355
